# Docling Heading to Chunk Trace
Audit heading detection and section metadata propagation into chunks using the current extraction and retrieval code.

In [ ]:
from __future__ import annotations
import json, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'playground' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUTPUT_DIR = ROOT / 'output'
candidates = sorted(OUTPUT_DIR.glob('*_hierarchy.json'), key=lambda p: p.stat().st_mtime, reverse=True)
if not candidates:
    raise FileNotFoundError('No *_hierarchy.json in output/.')
HIERARCHY_FILE = candidates[0]
DOC_ID = HIERARCHY_FILE.name.replace('_hierarchy.json', '')
COMPLETE_FILE = OUTPUT_DIR / f'{DOC_ID}_complete.json'

print('HIERARCHY_FILE:', HIERARCHY_FILE)
print('DOC_ID:', DOC_ID)
print('COMPLETE_FILE exists:', COMPLETE_FILE.exists())

In [ ]:
from backend.rag.retrieval.chunking.section_chunker import SectionChunker

hier_raw = json.loads(HIERARCHY_FILE.read_text(encoding='utf-8'))
hier = hier_raw.get('hierarchy', hier_raw)
sections = hier.get('sections', [])
sections_by_id = {s['section_id']: s for s in sections if isinstance(s, dict) and s.get('section_id')}

def path_titles(sid: str) -> list[str]:
    out, seen = [], set()
    while sid and sid not in seen and sid in sections_by_id:
        seen.add(sid)
        out.insert(0, str(sections_by_id[sid].get('title') or ''))
        sid = sections_by_id[sid].get('parent_id')
    return out

def path_ids(sid: str) -> list[str]:
    out, seen = [], set()
    while sid and sid not in seen and sid in sections_by_id:
        seen.add(sid)
        out.insert(0, sid)
        sid = sections_by_id[sid].get('parent_id')
    return out

expected_df = pd.DataFrame([
    {'section_id': s.get('section_id'), 'title': s.get('title'), 'expected_path': path_titles(s.get('section_id')), 'expected_path_ids': path_ids(s.get('section_id'))}
    for s in sections if isinstance(s, dict) and s.get('section_id')
])

if not COMPLETE_FILE.exists():
    raise FileNotFoundError(COMPLETE_FILE)
complete = json.loads(COMPLETE_FILE.read_text(encoding='utf-8'))
tb = pd.DataFrame((complete.get('extracted_elements') or {}).get('text_blocks') or [])
if not tb.empty:
    tb = tb[[c for c in ['reading_order', 'page', 'label', 'section_id', 'section_title', 'text'] if c in tb.columns]].copy()
    if 'reading_order' in tb.columns:
        tb = tb.sort_values('reading_order', kind='stable')
    tb['label_norm'] = tb.get('label', '').astype(str).str.lower().str.strip()
    tb['is_heading_like'] = tb['label_norm'].isin(['section_header', 'title'])
    tb['heading_text'] = tb['text'].where(tb['is_heading_like'], None)
    tb['nearest_prev_heading_text'] = tb['heading_text'].ffill()
    n = lambda x: ' '.join(str(x or '').split()).strip().lower()
    tb['heading_match'] = tb.get('section_title', '').map(n) == tb['nearest_prev_heading_text'].map(n)
    print('Text blocks:', len(tb))
    print('Heading-like blocks:', int(tb['is_heading_like'].sum()))
    print('Non-heading heading-match rate:', float(tb[~tb['is_heading_like']]['heading_match'].mean()) if (~tb['is_heading_like']).any() else 1.0)
    display(tb.head(40))

chunker = SectionChunker()
chunks = chunker.chunk_document(hierarchy_json_path=HIERARCHY_FILE, output_dir=OUTPUT_DIR, pdf_path=None)
chunk_df = pd.DataFrame([c.model_dump() for c in chunks])
print('Chunks produced:', len(chunk_df))

if not chunk_df.empty:
    join_df = chunk_df.merge(expected_df, how='left', on='section_id')
    n = lambda x: ' '.join(str(x or '').split()).strip().lower()
    join_df['title_match'] = join_df.apply(lambda r: n(r.get('section_title')) == n(r.get('title')), axis=1)
    join_df['path_match'] = join_df.apply(lambda r: (r.get('section_path') or []) == (r.get('expected_path') or []), axis=1)
    join_df['path_ids_match'] = join_df.apply(lambda r: (r.get('section_path_ids') or []) == (r.get('expected_path_ids') or []), axis=1)
    print('Title match rate:', float(join_df['title_match'].mean()))
    print('Path match rate:', float(join_df['path_match'].mean()))
    print('Path IDs match rate:', float(join_df['path_ids_match'].mean()))
    mism = join_df[~(join_df['title_match'] & join_df['path_match'] & join_df['path_ids_match'])]
    print('Mismatches:', len(mism))
    display(mism[[c for c in ['chunk_index','chunk_level','section_id','section_title','title','section_path','expected_path','section_path_ids','expected_path_ids'] if c in mism.columns]].head(40))

In [ ]:
audit_dir = ROOT / 'playground'
audit_dir.mkdir(parents=True, exist_ok=True)
chunk_csv = audit_dir / f'{DOC_ID}_chunk_section_audit.csv'
heading_csv = audit_dir / f'{DOC_ID}_docling_heading_assignment_audit.csv'

if 'chunk_df' in globals() and not chunk_df.empty:
    chunk_df[[c for c in ['chunk_index','chunk_level','content_type','section_id','section_title','section_path','section_path_ids','page_start','page_end'] if c in chunk_df.columns]].to_csv(chunk_csv, index=False)
    print('Saved:', chunk_csv)

if 'tb' in globals() and not tb.empty:
    tb[[c for c in ['reading_order','page','label','section_id','section_title','nearest_prev_heading_text','heading_match','text'] if c in tb.columns]].to_csv(heading_csv, index=False)
    print('Saved:', heading_csv)